# Validation externe sur corpus BnF étiqueté (exploratoire)

**Déplacé dans `04_exploration/`** depuis `classification_bois_cuivre/` — cette méthode
s'est révélée moins fiable que prévu comme mesure de référence (voir "Notes d'interprétation"
en fin de notebook) : le champ technique BnF contient des erreurs visibles à l'inspection, et
le corpus retéléchargé change de composition à chaque exécution, rendant les résultats peu
reproductibles d'un run à l'autre. Conservé ici à titre exploratoire ; l'évaluation de
référence du classifieur bois/cuivre se fait désormais via
`classification_bois_cuivre/05_evaluation_versions.ipynb` (annotations manuelles de
Céline Bohnert).

Test des versions du modèle **ResNet50** (bois / cuivre) sur un corpus
**indépendant et étiqueté par la BnF**.

Les illustrations d'Ovide proviennent de monographies : leur champ technique n'est pas
renseigné. Les images récupérées ici via l'API Gallica Images portent, elles, une
étiquette `properties_technical_process` — une **vérité terrain** qui permet de comparer
directement la prédiction du modèle à l'étiquette officielle.

| Classe | Étiquette(s) BnF |
|--------|------------------|
| Bois | `gravure sur bois` |
| Cuivre | `eau-forte`, `burin` |

**Principe du notebook :** on télécharge les images en local, puis on reconstruit le jeu
de test **à partir des fichiers réellement présents sur le disque**. Les prédictions
portent donc exactement sur les images téléchargées — aucun décalage possible.

*Profondeur `notebooks/04_exploration/` — `RACINE` remonte de deux niveaux, inchangé depuis le déplacement.*

## 1 · Imports et configuration

In [2]:
import os, sys, requests
import pandas as pd
import torch
from io import BytesIO
from PIL import Image

RACINE = os.path.abspath("../../")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)

from gallica_utils import charger_resnet, TRANSFORM_PRED, CLASSES, DEVICE

BASE_URL = "https://galimages-search.bnf.fr"
print("Imports OK")
print("RACINE :", RACINE)
print("Device :", DEVICE)

Imports OK
RACINE : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir
Device : cuda


## 2 · Paramètres

100 images par classe, datées 1500–1750 (même période que le corpus Ovide).

In [4]:
PROC_BOIS    = ["gravure sur bois"]
PROC_CUIVRE  = ["eau-forte", "burin"]

N_PAR_CLASSE = 500
DATE_MIN     = "1550"
DATE_MAX     = "1750"

DOSSIER_TEST = os.path.join(RACINE, "data", "editions_ovide", "test_bnf_etiquete")
os.makedirs(os.path.join(DOSSIER_TEST, "bois"),   exist_ok=True)
os.makedirs(os.path.join(DOSSIER_TEST, "cuivre"), exist_ok=True)

print("Destination des images :", DOSSIER_TEST)

Destination des images : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/editions_ovide/test_bnf_etiquete


## 3 · Récupération des images étiquetées

Interrogation de `/api/search` avec le filtre `filter_properties_technical_process`.
On collecte l'`ark` et le `link` IIIF de chaque image.

In [5]:
def recuperer_images(procedes, verite, n, date_min=DATE_MIN, date_max=DATE_MAX):
    """Récupère n images étiquetées avec l'un des procédés donnés."""
    r = requests.post(
        f"{BASE_URL}/api/search",
        json={
            "type": "textuelle",
            "rows": n,
            "filter_properties_technical_process": procedes,
            "search_after_context_date": date_min,
            "search_before_context_date": date_max,
        },
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    r.raise_for_status()
    docs = r.json()["response"]["docs"]
    return [
        {
            "ark":         d.get("ark"),          # ark de l'IMAGE (préfixe bfkfk...)
            "context_ark": d.get("context_ark"),  # ark du DOCUMENT Gallica parent
            "view_number": d.get("view_number"),  # folio dans le document
            "link":        d.get("link"),
            "verite":      verite,
        }
        for d in docs if d.get("link") and d.get("ark")
    ]


images = (
    recuperer_images(PROC_BOIS,   "bois",   N_PAR_CLASSE)
    + recuperer_images(PROC_CUIVRE, "cuivre", N_PAR_CLASSE)
)

nb_bois   = sum(1 for i in images if i["verite"] == "bois")
nb_cuivre = sum(1 for i in images if i["verite"] == "cuivre")
print(f"{len(images)} images récupérées  ({nb_bois} bois, {nb_cuivre} cuivre)")

1000 images récupérées  (500 bois, 500 cuivre)


## 4 · Téléchargement local

Chaque image est enregistrée sous `data/editions_ovide/test_bnf_etiquete/{classe}/{ark}.jpg`.
Le téléchargement est robuste : une image qui échoue n'interrompt pas la boucle.
Les images déjà présentes sont ignorées (relançable).

En plus des images, on sauvegarde `metadata.csv` (ark image → ark document +
folio) à côté — nécessaire pour reconstruire un lien Gallica valide en section 5bis,
indépendamment de la session en cours (l'ark image seul, préfixe `bfkfk...`, n'est
pas résolvable directement sur `gallica.bnf.fr`).

In [7]:
def vignette(url):
    """URL IIIF en largeur 400 px."""
    return url.replace("/max/0/", "/400,/0/")


CHEMIN_METADATA = os.path.join(DOSSIER_TEST, "metadata.csv")

reussis         = {"bois": 0, "cuivre": 0}
lignes_metadata = []
for k, img in enumerate(images, 1):
    print(f"  Téléchargement {k}/{len(images)}", end="\r")
    chemin = os.path.join(DOSSIER_TEST, img["verite"], f"{img['ark']}.jpg")

    if not os.path.exists(chemin):
        try:
            r = requests.get(vignette(img["link"]), timeout=30)
            r.raise_for_status()
            Image.open(BytesIO(r.content)).convert("RGB").save(chemin)
        except Exception:
            continue

    reussis[img["verite"]] += 1
    lignes_metadata.append({
        "ark":         img["ark"],
        "context_ark": img["context_ark"],
        "view_number": img["view_number"],
    })

print(f"\nTéléchargées — bois : {reussis['bois']}, cuivre : {reussis['cuivre']}")

# Métadonnées persistées à côté des images (voir section 5bis pour leur usage) —
# fusionnées avec celles déjà présentes, dédupliquées par ark.
df_meta = pd.DataFrame(lignes_metadata)
if os.path.exists(CHEMIN_METADATA):
    df_meta = pd.concat([pd.read_csv(CHEMIN_METADATA), df_meta]).drop_duplicates("ark")
df_meta.to_csv(CHEMIN_METADATA, index=False)
print(f"✓ Métadonnées sauvegardées : {CHEMIN_METADATA} ({len(df_meta)} entrées)")

  Téléchargement 1000/1000
Téléchargées — bois : 455, cuivre : 479
✓ Métadonnées sauvegardées : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/editions_ovide/test_bnf_etiquete/metadata.csv (934 entrées)


## 5 · Reconstruction du jeu de test depuis le disque

**Étape clé.** On ne se fie pas à la liste en mémoire mais aux **fichiers réellement
présents** : on parcourt les deux dossiers et on bâtit le DataFrame à partir d'eux.
Les prédictions porteront donc exactement sur les images téléchargées.

In [10]:
lignes = []
for verite in ["bois", "cuivre"]:
    dossier = os.path.join(DOSSIER_TEST, verite)
    for fichier in sorted(os.listdir(dossier)):
        if fichier.endswith(".jpg"):
            lignes.append({
                "ark":    fichier[:-4],
                "verite": verite,
                "chemin": os.path.join(dossier, fichier),
            })

df = pd.DataFrame(lignes)
print(f"{len(df)} images sur le disque")
print(df["verite"].value_counts().to_string())
df.head()

2164 images sur le disque
verite
cuivre    1448
bois       716


,ark,verite,chemin
0,bfkfk15088c,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...
1,bfkfk167hkt,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...
2,bfkfk167hq6,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...
3,bfkfk167hvk,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...
4,bfkfk167hwx,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...


## 5bis · Échantillon pour relecture experte

Avant d'aller plus loin (chargement des modèles, prédictions), on constitue un
échantillon aléatoire à envoyer à Céline pour qu'elle juge si l'étiquette technique
BnF (`properties_technical_process`) est fiable — indépendamment de tout modèle.

Génère une page HTML autonome (images encodées en base64, un seul fichier à
envoyer) : une vignette + l'étiquette BnF + un lien Gallica par image.

In [12]:
import random
import base64

N_ECHANTILLON = 50  # par classe
DOSSIER_RES   = os.path.join(RACINE, "resultats", "evaluation_modeles", "bois_cuivre")

# Métadonnées persistées en section 4 (ark image → ark document + folio) —
# vérifié via l'API : /api/search renvoie directement `context_ark` (ark du
# document Gallica) et `view_number` (folio), pas besoin de parser `link`.
df_meta = pd.read_csv(os.path.join(DOSSIER_TEST, "metadata.csv")).set_index("ark")


def lien_gallica(ark):
    """
    Construit le lien Gallica du document source à partir des métadonnées.
    L'ark IMAGE renvoyé par /api/search (préfixe bfkfk...) n'est pas résolvable
    directement sur gallica.bnf.fr (403 Access Interdit) — il faut l'ark du
    DOCUMENT parent (`context_ark`) et le folio (`view_number`).
    Retombe sur l'ark image si l'entrée est absente de metadata.csv (image
    téléchargée avant l'introduction de ce fichier).
    """
    if ark not in df_meta.index:
        return f"https://gallica.bnf.fr/ark:/12148/{ark}"
    ligne = df_meta.loc[ark]
    return f"https://gallica.bnf.fr/ark:/12148/{ligne['context_ark']}/f{int(ligne['view_number'])}.item"


random.seed(42)
echantillons = {}
for verite in ["bois", "cuivre"]:
    sous_ensemble = df[df["verite"] == verite]
    n = min(N_ECHANTILLON, len(sous_ensemble))
    echantillons[verite] = sous_ensemble.sample(n, random_state=42).reset_index(drop=True)

print(f"Échantillon : {sum(len(v) for v in echantillons.values())} images — "
      f"{ {k: len(v) for k, v in echantillons.items()} }")


def image_vers_base64(chemin, largeur=250):
    """Redimensionne et encode une image en base64 pour l'intégrer directement dans le HTML."""
    img   = Image.open(chemin).convert("RGB")
    ratio = largeur / img.width
    img   = img.resize((largeur, int(img.height * ratio)))
    buf   = BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


def bloc_classe(verite, sous_df):
    """Construit la section HTML (titre + grille de cartes) pour une classe donnée."""
    cartes = []
    for _, row in sous_df.iterrows():
        b64 = image_vers_base64(row["chemin"])
        cartes.append(f"""
        <div class="carte">
          <img src="data:image/jpeg;base64,{b64}">
          <div class="legende">
            <b>{row['verite'].upper()}</b><br>
            <a href="{lien_gallica(row['ark'])}" target="_blank">voir sur Gallica</a>
          </div>
        </div>""")
    return f"""
    <h2>{verite.upper()} — {len(sous_df)} images</h2>
    <div class="grille">
      {"".join(cartes)}
    </div>"""

# Cuivre en premier, bois ensuite — deux blocs bien séparés
sections = bloc_classe("cuivre", echantillons["cuivre"]) + bloc_classe("bois", echantillons["bois"])

html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Échantillon — étiquette technique BnF (bois/cuivre)</title>
<style>
  body {{ font-family: sans-serif; background: #f8f5f0; margin: 20px; }}
  h1 {{ font-size: 18px; }}
  h2 {{ font-size: 15px; margin-top: 30px; border-bottom: 2px solid #ccc; padding-bottom: 4px; }}
  .grille {{ display: flex; flex-wrap: wrap; gap: 12px; }}
  .carte {{ width: 260px; background: white; border: 1px solid #ddd; border-radius: 6px; padding: 8px; }}
  .carte img {{ width: 100%; border-radius: 4px; }}
  .legende {{ font-size: 12px; margin-top: 6px; }}
</style></head>
<body>
<h1>Échantillon pour relecture — étiquette technique BnF ({N_ECHANTILLON}/classe)</h1>
<p>Étiquette BnF (<code>properties_technical_process</code>) affichée sous chaque image.
Objectif : vérifier si l'étiquette correspond visuellement à la technique de gravure réellement représentée.
Les deux classes sont regroupées séparément ci-dessous.</p>
{sections}
</body></html>"""

os.makedirs(DOSSIER_RES, exist_ok=True)
chemin_html = os.path.join(DOSSIER_RES, "echantillon_verite_bnf.html")
with open(chemin_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Échantillon HTML généré : {chemin_html}")

Échantillon : 100 images — {'bois': 50, 'cuivre': 50}
✓ Échantillon HTML généré : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/evaluation_modeles/bois_cuivre/echantillon_verite_bnf.html


## 6 · Chargement des modèles

Les fichiers `.pth` sont dans `modeles/bois_cuivre/`.
**Vérifier les noms ci-dessous** et corriger si nécessaire.

In [23]:
DOSSIER_MODELES = os.path.join(RACINE, "modeles", "bois_cuivre")

# v1/v2/v3 retires temporairement pour accelerer (deja connus sur ce meme corpus BnF :
# 60.3% / 65.2% / 56.7%, voir resume precedent) — focus sur v4.0.0 / v4.1.0 / v4.1.1
fichiers_modeles = {
    "v4.0.0": "resnet50_bois_cuivre_v4.0.0.pth",
    "v4.1.0": "resnet50_bois_cuivre_v4.1.0.pth",
    "v4.1.1": "resnet50_bois_cuivre_v4.1.1.pth",
}

modeles = {
    v: charger_resnet(os.path.join(DOSSIER_MODELES, nom))
    for v, nom in fichiers_modeles.items()
}

✓ ResNet50 chargé : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/modeles/bois_cuivre/resnet50_bois_cuivre_v4.0.0.pth
✓ ResNet50 chargé : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/modeles/bois_cuivre/resnet50_bois_cuivre_v4.1.0.pth
✓ ResNet50 chargé : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/modeles/bois_cuivre/resnet50_bois_cuivre_v4.1.1.pth


## 7 · Prédiction des versions sur les fichiers locaux

On lit chaque image **depuis le disque** (et non depuis une URL), avec exactement
les mêmes transformations que `predire_technique`.

In [24]:
def predire_fichier_local(chemin, modele, device=DEVICE):
    """Prédit (classe, confiance) à partir d'un fichier image local."""
    try:
        img        = Image.open(chemin).convert("RGB")
        img_tensor = TRANSFORM_PRED(img).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = modele(img_tensor)
            probs   = torch.softmax(outputs, dim=1)
            _, pred = torch.max(outputs, 1)
        return CLASSES[pred.item()], round(probs[0][pred.item()].item(), 3)
    except Exception:
        return "inconnu", 0.0


print("Prédiction en cours…")
for k, row in df.iterrows():
    print(f"  {k + 1}/{len(df)}", end="\r")
    for v, modele in modeles.items():
        classe, conf = predire_fichier_local(row["chemin"], modele)
        df.loc[k, f"pred_{v}"] = classe
        df.loc[k, f"conf_{v}"] = conf

print(f"\n{len(df)} images classées par les 3 versions")
print("Colonnes ajoutées :", [c for c in df.columns if c.startswith(("pred_", "conf_"))])
df.head()

Prédiction en cours…
  748/748
748 images classées par les 3 versions
Colonnes ajoutées : ['pred_v4.0.0', 'conf_v4.0.0', 'pred_v4.1.0', 'conf_v4.1.0', 'pred_v4.1.1', 'conf_v4.1.1']


,ark,verite,chemin,pred_v4.0.0,conf_v4.0.0,pred_v4.1.0,conf_v4.1.0,pred_v4.1.1,conf_v4.1.1
0,bfkfk1h76b,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...,cuivre,1.000,cuivre,0.996,cuivre,1.000
1,bfkfk1h7cw,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...,cuivre,1.000,cuivre,1.000,cuivre,1.000
2,bfkfk1h7h4,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...,cuivre,1.000,cuivre,1.000,cuivre,1.000
3,bfkfk1h7v7,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...,cuivre,0.990,cuivre,0.987,cuivre,0.998
4,bfkfk21wrg6,bois,/mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/wor...,bois,0.976,bois,0.778,bois,0.986


## 8 · Résultats — accuracy de chaque version

Accuracy globale et détail par classe, calculés **uniquement sur les images locales**.

In [25]:
print("=" * 60)
print("VALIDATION SUR CORPUS BnF ÉTIQUETÉ")
print(f"{len(df)} images  |  Bois = « gravure sur bois »  |  Cuivre = « eau-forte » + « burin »")
print("=" * 60)

resume = []
for v in fichiers_modeles.keys():
    valides = df[df[f"pred_{v}"] != "inconnu"]
    n       = len(valides)
    correct = (valides[f"pred_{v}"] == valides["verite"]).sum()

    bois   = valides[valides["verite"] == "bois"]
    cuivre = valides[valides["verite"] == "cuivre"]
    acc_bois   = (bois[f"pred_{v}"]   == "bois").sum()
    acc_cuivre = (cuivre[f"pred_{v}"] == "cuivre").sum()

    print(f"\n  {v.upper()}  —  accuracy globale : {correct}/{n} = {correct / n:.1%}")
    print(f"        bois   : {acc_bois}/{len(bois)} ({acc_bois / len(bois):.1%})")
    print(f"        cuivre : {acc_cuivre}/{len(cuivre)} ({acc_cuivre / len(cuivre):.1%})")

    resume.append({
        "version":         v,
        "accuracy":        round(correct / n, 3),
        "bois_ok":         f"{acc_bois}/{len(bois)}",
        "cuivre_ok":       f"{acc_cuivre}/{len(cuivre)}",
        "images_classees": n,
    })

df_resume = pd.DataFrame(resume)
print()
print(df_resume.to_string(index=False))

VALIDATION SUR CORPUS BnF ÉTIQUETÉ
748 images  |  Bois = « gravure sur bois »  |  Cuivre = « eau-forte » + « burin »

  V4.0.0  —  accuracy globale : 528/748 = 70.6%
        bois   : 56/261 (21.5%)
        cuivre : 472/487 (96.9%)

  V4.1.0  —  accuracy globale : 508/748 = 67.9%
        bois   : 37/261 (14.2%)
        cuivre : 471/487 (96.7%)

  V4.1.1  —  accuracy globale : 513/748 = 68.6%
        bois   : 36/261 (13.8%)
        cuivre : 477/487 (97.9%)

version  accuracy bois_ok cuivre_ok  images_classees
 v4.0.0     0.706  56/261   472/487              748
 v4.1.0     0.679  37/261   471/487              748
 v4.1.1     0.686  36/261   477/487              748


## 9 · Sauvegarde

Détail par image et résumé dans `resultats/evaluation_modeles/bois_cuivre/`.

In [26]:
DOSSIER_RES = os.path.join(RACINE, "resultats", "evaluation_modeles", "bois_cuivre")
os.makedirs(DOSSIER_RES, exist_ok=True)

chemin_detail = os.path.join(DOSSIER_RES, "validation_bnf_etiquete_detail.csv")
chemin_resume = os.path.join(DOSSIER_RES, "validation_bnf_etiquete_resume.csv")

df.to_csv(chemin_detail, index=False)
df_resume.to_csv(chemin_resume, index=False)

print("Détail :", chemin_detail)
print("Résumé :", chemin_resume)

Détail : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/evaluation_modeles/bois_cuivre/validation_bnf_etiquete_detail.csv
Résumé : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/evaluation_modeles/bois_cuivre/validation_bnf_etiquete_resume.csv


## Notes d'interprétation

- **Validation externe** : images indépendantes du corpus d'entraînement et de la
  validation de C. Bohnert, étiquetées par la BnF.
- Le corpus est **plus varié** que les illustrations d'Ovide (sujets, styles, mises en
  page différents). Une accuracy plus basse qu'en validation interne est **attendue** et
  mesure la capacité de généralisation.
- Un écart marqué entre la reconnaissance du **bois** et du **cuivre** révèle le biais de
  chaque version (tendance à sur-prédire une classe sur des images hors domaine).
- Ce corpus BnF étiqueté constitue aussi un **réservoir d'entraînement** pour une future
  version, plus large et plus varié que le corpus actuel.

**Deux limites méthodologiques découvertes en l'utilisant (raisons du déplacement en exploration) :**

1. **Étiquette technique peu fiable** — à l'inspection visuelle, une partie des images
   étiquetées "bois" (`properties_technical_process = gravure sur bois`) ne ressemblent pas
   à de vraies gravures sur bois. Les écarts de rappel bois observés ici (souvent 15-30 %,
   bien plus bas que sur le test interne honnête des versions v4, ~95-98 %) reflètent donc en
   partie du bruit d'étiquetage, pas uniquement des erreurs du modèle.
2. **Corpus non stable d'un run à l'autre** — la cellule 3-4 réinterroge l'API et retélécharge
   à chaque exécution ; l'échantillon renvoyé (nombre et identité des images) change à chaque
   fois, ce qui fait bouger sensiblement les pourcentages sans changement du modèle. Comparer
   deux versions sur deux exécutions différentes de ce notebook n'est donc pas fiable.

Pour ces raisons, ce notebook n'est plus utilisé comme mesure de référence pour choisir entre
versions du classifieur — voir `classification_bois_cuivre/05_evaluation_versions.ipynb`
(basé sur les annotations manuelles de Céline Bohnert, un dénominateur d'images fixe) pour
l'évaluation qui a servi à clôturer l'axe.